# 03 — WoE Encoding & Information Value

This notebook demonstrates the Weight of Evidence (WoE) transformation and Information Value (IV)  
computation using the **optbinning** library. We visualise binning tables and filter features  
by predictive strength (IV > 0.10).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from optbinning import OptimalBinning

import config
from src.woe_iv import compute_woe_iv, filter_by_iv, CANDIDATE_FEATURES

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
%matplotlib inline

## 3.1 Load Engineered Features

In [ ]:
df = pd.read_csv(config.FEATURES_CSV)
print(f"Shape: {df.shape}")
print(f"Target distribution:\n{df['bad_loan'].value_counts()}")

## 3.2 WoE / IV Computation

We use `OptimalBinning` from the **optbinning** library, which performs automatic  
monotonic binning using constraint programming (CP solver).

In [ ]:
woe_df, iv_table, binning_objects = compute_woe_iv(df)

print("\n" + "=" * 45)
print("  INFORMATION VALUE SUMMARY")
print("=" * 45)
print(iv_table.to_string(index=False))

## 3.3 IV Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

colors = ["#2ecc71" if iv > 0.3 else "#3498db" if iv > 0.1 else "#e74c3c"
          for iv in iv_table["iv"]]

ax.barh(iv_table["feature"], iv_table["iv"], color=colors, edgecolor="white")
ax.axvline(x=config.IV_THRESHOLD, color="red", linestyle="--", linewidth=1.5,
           label=f"IV threshold = {config.IV_THRESHOLD}")
ax.set_xlabel("Information Value", fontsize=12)
ax.set_title("Information Value by Feature", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3.4 Binning Tables for Top Features

Let's inspect the optimal bins, event rates, and WoE values for the top predictors.

In [ ]:
# Show binning table for the top 3 IV features
top_features = iv_table.head(3)["feature"].tolist()

for feat in top_features:
    if feat in binning_objects:
        print(f"\n{'='*50}")
        print(f"  Binning Table: {feat}")
        print(f"{'='*50}")
        bt = binning_objects[feat].binning_table.build()
        print(bt.to_string())
        print()

## 3.5 WoE Plots for Top Features

In [ ]:
fig, axes = plt.subplots(1, min(3, len(top_features)), figsize=(6 * min(3, len(top_features)), 5))
if len(top_features) == 1:
    axes = [axes]

for ax, feat in zip(axes, top_features):
    if feat in binning_objects:
        bt = binning_objects[feat].binning_table.build()
        # Exclude Totals and Missing/Special rows for cleaner plot
        bt_plot = bt.iloc[:-3] if len(bt) > 3 else bt
        woe_vals = bt_plot["WoE"].values.astype(float)
        bin_labels = bt_plot["Bin"].astype(str) if "Bin" in bt_plot.columns else bt_plot.index.astype(str)
        colors = ["#e74c3c" if w < 0 else "#2ecc71" for w in woe_vals]
        ax.barh(range(len(woe_vals)), woe_vals, color=colors, edgecolor="white")
        ax.set_yticks(range(len(woe_vals)))
        ax.set_yticklabels(bin_labels, fontsize=9)
        ax.axvline(x=0, color="black", linewidth=0.5)
        ax.set_title(f"WoE – {feat}", fontsize=12, fontweight="bold")
        ax.set_xlabel("WoE")

plt.tight_layout()
plt.show()

## 3.6 Filter Features by IV Threshold

In [ ]:
filtered_woe, filtered_iv = filter_by_iv(woe_df, iv_table)

print(f"Features retained (IV > {config.IV_THRESHOLD}):")
print(filtered_iv.to_string(index=False))
print(f"\nFiltered WoE matrix shape: {filtered_woe.shape}")

## 3.7 WoE Feature Correlation

In [ ]:
corr = filtered_woe.drop(columns=["bad_loan"]).corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title("WoE Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3.8 Key Takeaways

- **WoE encoding** converts each numeric feature into monotonic bins aligned with log-odds of default.
- **IV** ranks features by predictive power; features with IV < 0.10 are dropped.
- Top predictors are expected to include: `fico_midpoint`, `int_rate`, `dti`, `revol_util`.
- WoE-transformed features are now ready for logistic regression in the scorecard model.

Proceed to **04_model_training.ipynb** →